# MSTR T028 — Ephemeral Weight Acquisition (Colab)

Founder-authorized acquisition of the **exact pinned artifacts** from the frozen
T027 manifest. Runs entirely inside this ephemeral VM:

- plain HTTPS GET, **no authentication**, no gated terms, **USD 0.00**
- per-file size + SHA-256 verified against the frozen manifest (fail closed)
- the machine-verifiable report is the only durable output; binary copies are
  deleted when this VM is reclaimed (`docs/canonical/STORAGE_ARCHITECTURE.md`)

**Step 1** — upload `artifacts/manifests/T027-weight-access.json` to the Colab
session files, then set `MANIFEST` below.

**Step 2** — run all cells. Reports land in `/content/reports/`; download them
back into the repo as evidence inputs.

In [ ]:
import sys
from pathlib import Path

REPO = Path("/content/mstr")  # adjust if cloned elsewhere
sys.path.insert(0, str(REPO / "src"))
MANIFEST = REPO / "artifacts/manifests/T027-weight-access.json"
OUT = Path("/content/reports")
OUT.mkdir(exist_ok=True)
print("manifest exists:", MANIFEST.exists())


In [ ]:
import json

from mstr_qualify.acquisition import build_acquisition_plan

manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
plan = build_acquisition_plan(manifest)
print(f"{len(plan)} pinned files across", sorted({f.candidate_id for f in plan}))
total = sum(f.expected_size_bytes for f in plan if f.expected_size_bytes)
print(f"total expected bytes: {total:,} ({total / 2**30:.2f} GiB)")


In [ ]:
# Acquire one candidate per invocation to bound peak disk usage.
CANDIDATE = "qwen3.5-2b"  # <- set per run; repeat for each candidate
script = REPO / "colab/mstr_t028_acquire.py"
report = OUT / f"{CANDIDATE}.json"
cmd = (
    f"python {script} --manifest {MANIFEST} --candidate {CANDIDATE} "
    f"--workdir /content/wt --report {report}"
)
!{cmd}


After every candidate report lands in `/content/reports/`, download them and
attach them to the T028 PR. Nothing else leaves this VM; binaries are deleted
by the runner itself immediately after verification.